# Notebook YOLO-World × CAPTURe — Baseline Clásico de Conteo

**Objetivo:** evaluar YOLO-World como baseline clásico de detección/conteo sobre CAPTURe real,
usando las mismas métricas y el mismo n=50 estratificado (semilla 42) que los VLMs ya evaluados.

**Modelo:** YOLO-World (yolov8m-worldv2) — open-vocabulary, recibe el nombre de la categoría
exactamente como los VLMs reciben la pregunta textual.

**Dataset:** CAPTURe real — 824 imágenes, objetos cotidianos, GT rango 8–907.

**Nota metodológica:** YOLO-World detecta objetos dado un texto de clase y devuelve bounding
boxes con score de confianza. El conteo = número de boxes que superan el umbral de confianza
tras NMS (Non-Maximum Suppression). La elección del umbral es el parámetro más sensible;
se evalúan tres valores para análisis de sensibilidad.

In [ ]:
# YOLO-World requiere ultralytics >= 8.1. scipy para intervalos de confianza.
!pip install -q "ultralytics>=8.1.0" huggingface_hub scipy

import ultralytics
print(f"ultralytics: {ultralytics.__version__}")

In [ ]:
import os, json, random, time, datetime, zipfile, re
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from scipy import stats

# ── Configuración de la corrida ──────────────────────────────────────────────
BENCHMARK_NAME   = "CAPTURe"
MODEL_TAG        = "yolov8m-worldv2"   # YOLO-World medium v2

SAMPLE_SIZE_REAL = 50          # None = dataset completo (824)
RANDOM_SEED      = 42

# Umbral principal de confianza (se prueban tres en análisis de sensibilidad)
CONF_MAIN   = 0.10   # más bajo = más detecciones; recomendado para conteo denso
CONF_MEDIUM = 0.25   # default ultralytics
CONF_HIGH   = 0.50   # conservador

IOU_NMS = 0.45   # umbral NMS (default ultralytics)

OUTPUT_TXT_PATH = f"/content/resultados_{BENCHMARK_NAME}_{MODEL_TAG}.txt"
OUTPUT_CSV_PATH = f"/content/detalle_{BENCHMARK_NAME}_{MODEL_TAG}.csv"

print(f"Modelo:   {MODEL_TAG}")
print(f"Muestra:  {SAMPLE_SIZE_REAL or 'completa (824)'} real | semilla {RANDOM_SEED}")
print(f"Conf principal: {CONF_MAIN}  |  medio: {CONF_MEDIUM}  |  alto: {CONF_HIGH}")
print(f"IOU NMS:  {IOU_NMS}")

In [ ]:
from ultralytics import YOLOWorld

print(f"Cargando {MODEL_TAG}...")
model = YOLOWorld(f"{MODEL_TAG}.pt")   # descarga automática si no existe

# Verificar GPU
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    model.to("cuda")
else:
    print("ADVERTENCIA: sin GPU. La inferencia será más lenta.")

print("Modelo listo.")

In [ ]:
from huggingface_hub import hf_hub_download

REAL_IMAGE_DIR = "/content/capture_real"

# Descargar solo si no existe ya
if not os.path.exists(REAL_IMAGE_DIR):
    print("Descargando real_dataset.zip...")
    real_zip = hf_hub_download(
        repo_id="atinp/CAPTURe",
        filename="real_dataset.zip",
        repo_type="dataset"
    )
    print("Extrayendo...")
    with zipfile.ZipFile(real_zip, "r") as z:
        z.extractall("/content/")
    import glob
    todas_las_imagenes = glob.glob("/content/**/*.jpg", recursive=True)
    print(f"Imágenes encontradas tras extraer: {len(todas_las_imagenes)}")
    if todas_las_imagenes:
        REAL_IMAGE_DIR = os.path.dirname(todas_las_imagenes[0])
        print(f"REAL_IMAGE_DIR ajustado a: {REAL_IMAGE_DIR}")
    else:
        print("ADVERTENCIA: no se encontraron imágenes .jpg tras extraer — revisar estructura del zip.")
else:
    print("Imágenes ya presentes.")

print("Descargando metadatos...")
real_meta = hf_hub_download(
    repo_id="atinp/CAPTURe",
    filename="real_metadata.json",
    repo_type="dataset"
)

with open(real_meta, "r") as f:
    real_metadata = json.load(f)

real_df = pd.DataFrame(real_metadata)
print(f"CAPTURe real: {len(real_df)} imágenes, columnas: {list(real_df.columns)}")
print(real_df.head(3))

In [ ]:
# Ajustar el directorio de imágenes según cómo se extrajo el zip.
# CAPTURe real extrae en /content/real/ con subcarpetas por categoría.
import glob

candidates = glob.glob("/content/**/real_dataset", recursive=True) + \
             glob.glob("/content/real*", recursive=False) + \
             glob.glob("/content/real", recursive=False)
print("Candidatos encontrados:", candidates)

# Buscar el directorio que contenga .jpg/.png
for c in candidates:
    imgs = glob.glob(os.path.join(c, "**", "*.jpg"), recursive=True) + \
           glob.glob(os.path.join(c, "**", "*.png"), recursive=True)
    if imgs:
        REAL_IMAGE_DIR = c
        print(f"Directorio de imágenes: {REAL_IMAGE_DIR} ({len(imgs)} imgs encontradas)")
        break

# Construir lookup: image_file → path completo
all_images = glob.glob(os.path.join(REAL_IMAGE_DIR, "**", "*"), recursive=True)
img_lookup = {os.path.basename(p): p for p in all_images if p.lower().endswith((".jpg", ".jpeg", ".png"))}
print(f"Lookup construido: {len(img_lookup)} imágenes indexadas")
print("Ejemplo:", list(img_lookup.items())[:2])

In [ ]:
# Muestreo estratificado por bucket de magnitud — reproduce exactamente el sample
# usado por todos los VLMs (RANDOM_SEED=42, mismas proporciones de bucket).

BUCKETS = {
    "8-15":   (8,   15),
    "16-28":  (16,  28),
    "29-67":  (29,  67),
    "68-907": (68, 907),
}

def asignar_bucket(gt):
    for nombre, (lo, hi) in BUCKETS.items():
        if lo <= gt <= hi:
            return nombre
    return "otro"

real_df["bucket"] = real_df["ground_truth"].apply(asignar_bucket)
real_df = real_df[real_df["bucket"] != "otro"].copy()

print("Distribución original por bucket:")
print(real_df["bucket"].value_counts().sort_index())

random.seed(RANDOM_SEED)

if SAMPLE_SIZE_REAL and SAMPLE_SIZE_REAL < len(real_df):
    bucket_counts = real_df["bucket"].value_counts()
    total = len(real_df)
    muestra = []
    asignados = 0
    buckets_ordenados = list(BUCKETS.keys())

    for i, bucket in enumerate(buckets_ordenados):
        n_bucket = len(real_df[real_df["bucket"] == bucket])
        if i == len(buckets_ordenados) - 1:
            n = SAMPLE_SIZE_REAL - asignados
        else:
            n = round(SAMPLE_SIZE_REAL * n_bucket / total)
        n = min(n, n_bucket)
        sample_bucket = real_df[real_df["bucket"] == bucket].sample(n=n, random_state=RANDOM_SEED)
        muestra.append(sample_bucket)
        asignados += n

    muestra_df = pd.concat(muestra).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
else:
    muestra_df = real_df.copy().reset_index(drop=True)

print(f"\nMuestra final: {len(muestra_df)} imágenes")
print(muestra_df["bucket"].value_counts().sort_index())
print("\nPrimeras filas:")
print(muestra_df[["image_file", "object", "ground_truth", "bucket"]].head(5))

In [ ]:
from IPython.display import display

print("Preview de 2 imágenes de la muestra:")
for _, row in muestra_df.sample(2, random_state=0).iterrows():
    img_path = img_lookup.get(row["image_file"])
    if img_path:
        img = Image.open(img_path).convert("RGB")
        print(f"  {row['object']} | GT={row['ground_truth']} | bucket={row['bucket']}")
        display(img.resize((400, 300)))
    else:
        print(f"  No encontrada: {row['image_file']}")

In [ ]:
def predecir_yolo_world(img_path: str, categoria: str, conf: float) -> dict:
    """
    Corre YOLO-World en img_path buscando `categoria`.
    Retorna: {pred, n_boxes, max_conf, mean_conf, boxes_xyxy}
    """
    # Limpiar categoria: YOLO-World funciona mejor con singular y sin puntuación extra
    clase = categoria.lower().strip().rstrip("s")  # simple singularización
    # Excepciones manuales comunes
    SINGULAR_MAP = {
        "bricke": "brick",         "watche": "watch",
        "candl": "candle",         "glasse": "glasses",
        "sunglasse": "sunglasses", "cupcake trae": "cupcake tray",
        "tomato": "tomato",        "pea": "peas",
        "pencil": "pencil",        "mini blind": "mini blind",
        "roof til": "roof tile",   "toilet paper roll": "toilet paper roll",
        "finger food": "finger food", "flower pot": "flower pot",
        "keyboard key": "keyboard key", "spoon": "spoon",
    }
    clase = SINGULAR_MAP.get(clase, clase)

    model.set_classes([clase])
    results = model.predict(
        source=img_path,
        conf=conf,
        iou=IOU_NMS,
        verbose=False,
        device=device,
    )
    boxes = results[0].boxes
    n = len(boxes) if boxes is not None else 0
    confs = boxes.conf.cpu().numpy() if (boxes is not None and len(boxes) > 0) else np.array([])
    xyxy = boxes.xyxy.cpu().numpy() if (boxes is not None and len(boxes) > 0) else np.zeros((0, 4))

    return {
        "pred":      n,
        "n_boxes":   n,
        "max_conf":  float(confs.max()) if len(confs) > 0 else 0.0,
        "mean_conf": float(confs.mean()) if len(confs) > 0 else 0.0,
        "boxes_xyxy": xyxy,
        "clase_usada": clase,
    }

print("Función predecir_yolo_world definida.")
print("Prueba rápida con primera imagen de la muestra...")
row0 = muestra_df.iloc[0]
path0 = img_lookup.get(row0["image_file"], "")
if path0:
    r = predecir_yolo_world(path0, row0["object"], CONF_MAIN)
    print(f"  objeto={row0['object']} | clase_usada={r['clase_usada']} | GT={row0['ground_truth']} | pred={r['pred']} boxes")
else:
    print("  Imagen no encontrada, ajustar img_lookup en Cell 5.")

In [ ]:
resultados = []

print(f"Iniciando inferencia: {len(muestra_df)} imágenes × 3 umbrales de confianza...\n")

for i, (_, row) in enumerate(muestra_df.iterrows()):
    img_path = img_lookup.get(row["image_file"])
    if not img_path:
        print(f"[{i+1}] SKIP — imagen no encontrada: {row['image_file']}")
        continue

    row_result = {
        "imagen":       row["image_file"],
        "categoria":    row["object"],
        "ground_truth": int(row["ground_truth"]),
        "bucket":       row["bucket"],
    }

    # Correr con los 3 umbrales (YOLO-World es rápido, ~0.05s/img en GPU)
    for conf_val, conf_key in [(CONF_MAIN, "pred_low"), (CONF_MEDIUM, "pred_mid"), (CONF_HIGH, "pred_high")]:
        t0 = time.time()
        r = predecir_yolo_world(img_path, row["object"], conf_val)
        elapsed = time.time() - t0
        row_result[conf_key]          = r["pred"]
        row_result[f"n_boxes_{conf_key}"] = r["n_boxes"]
        row_result[f"time_{conf_key}"] = elapsed
        row_result[f"clase_usada"]    = r["clase_usada"]

    # Métricas con umbral principal (CONF_MAIN)
    row_result["prediccion"]  = row_result["pred_low"]
    row_result["error_abs"]   = abs(row_result["pred_low"] - row_result["ground_truth"])
    row_result["parse_fail"]  = False   # YOLO no tiene parse fails
    row_result["timeout"]     = False

    resultados.append(row_result)

    if (i + 1) % 10 == 0 or i == 0:
        print(f"[{i+1}/{len(muestra_df)}] {row['object']:25s} | GT={row['ground_truth']:4d} "
              f"| pred(c={CONF_MAIN})={row_result['pred_low']:3d} "
              f"| pred(c={CONF_MEDIUM})={row_result['pred_mid']:3d} "
              f"| pred(c={CONF_HIGH})={row_result['pred_high']:3d}")

df = pd.DataFrame(resultados)
print(f"\nInferencia completa: {len(df)} filas.")

In [ ]:
# ── Métricas globales por umbral ────────────────────────────────────────────
print("=" * 65)
print(f"RESULTADOS — CAPTURe real — {MODEL_TAG}")
print("=" * 65)

for conf_val, col, label in [
    (CONF_MAIN,   "pred_low",  f"conf={CONF_MAIN} (principal)"),
    (CONF_MEDIUM, "pred_mid",  f"conf={CONF_MEDIUM}"),
    (CONF_HIGH,   "pred_high", f"conf={CONF_HIGH}"),
]:
    errs = (df[col] - df["ground_truth"]).abs()
    acc  = (df[col] == df["ground_truth"]).mean()
    mae  = errs.mean()
    print(f"\n  [{label}]")
    print(f"    Accuracy exacta : {acc:.4f}  ({acc*100:.1f}%)")
    print(f"    MAE             : {mae:.4f}")
    print(f"    Varianza error  : {errs.var():.2f}")

print()

# ── Intervalos de confianza (umbral principal) ───────────────────────────────
print("\n── Intervalos de confianza 95% (umbral principal conf={}) ──".format(CONF_MAIN))

# Wilson CI para accuracy
n   = len(df)
acc = (df["pred_low"] == df["ground_truth"]).mean()
z   = 1.96
denom   = 1 + z**2 / n
center  = (acc + z**2 / (2*n)) / denom
margin  = (z * np.sqrt(acc*(1-acc)/n + z**2/(4*n**2))) / denom
ci_acc_lo = max(0, center - margin)
ci_acc_hi = min(1, center + margin)
print(f"  Accuracy (Wilson):  {acc:.3f}  [{ci_acc_lo:.3f}, {ci_acc_hi:.3f}]")

# Bootstrap CI para MAE
errs = (df["pred_low"] - df["ground_truth"]).abs().values
np.random.seed(42)
boot_maes = [np.random.choice(errs, size=len(errs), replace=True).mean() for _ in range(2000)]
ci_mae_lo, ci_mae_hi = np.percentile(boot_maes, [2.5, 97.5])
print(f"  MAE (bootstrap):    {errs.mean():.2f}  [{ci_mae_lo:.2f}, {ci_mae_hi:.2f}]")

In [ ]:
# ── Desglose por bucket de magnitud (umbral principal) ───────────────────────
print("\n── Desglose por bucket (conf={}) ──".format(CONF_MAIN))
print(f"{'Bucket':10s} {'n':>4s}  {'Accuracy':>9s}  {'MAE':>8s}  {'VLM_mejor_ref':>14s}")
print("-" * 55)

# Referencia VLM: Qwen2-VL (mejor en CAPTURe real n=50)
VLM_REF = {
    "8-15":   (0.3333, 2.92),
    "16-28":  (0.0000, 7.30),
    "29-67":  (0.0000, 20.9),
    "68-907": (0.0625, 52.9),
}

for bucket in ["8-15", "16-28", "29-67", "68-907"]:
    sub = df[df["bucket"] == bucket]
    if len(sub) == 0:
        continue
    errs_b = (sub["pred_low"] - sub["ground_truth"]).abs()
    acc_b  = (sub["pred_low"] == sub["ground_truth"]).mean()
    mae_b  = errs_b.mean()
    ref_acc, ref_mae = VLM_REF.get(bucket, (None, None))
    ref_str = f"acc={ref_acc:.2f} mae={ref_mae:.1f}" if ref_acc is not None else "—"
    print(f"  {bucket:8s}  {len(sub):4d}  {acc_b:.4f}     {mae_b:8.2f}  [{ref_str}]")

In [ ]:
# ── Desglose por categoría — qué objetos cuenta mejor/peor YOLO-World ─────
print("\n── Desglose por categoría (conf={}) ──".format(CONF_MAIN))

cat_stats = df.groupby("categoria").apply(lambda g: pd.Series({
    "n":        len(g),
    "gt_mean":  g["ground_truth"].mean(),
    "pred_mean": g["pred_low"].mean(),
    "mae":      (g["pred_low"] - g["ground_truth"]).abs().mean(),
    "accuracy": (g["pred_low"] == g["ground_truth"]).mean(),
})).reset_index().sort_values("mae")

print(f"\n{'Categoría':22s} {'n':>3s}  {'GT_med':>6s}  {'Pred_med':>8s}  {'MAE':>7s}  {'Acc':>5s}")
print("-" * 60)
for _, r in cat_stats.iterrows():
    print(f"  {r['categoria']:20s} {r['n']:3.0f}  {r['gt_mean']:6.1f}   {r['pred_mean']:8.1f}   {r['mae']:7.2f}  {r['accuracy']:5.2f}")

# Mejores y peores
print("\n▲ Categorías con menor MAE (YOLO cuenta mejor):")
print(cat_stats.head(3)[["categoria", "n", "mae", "accuracy"]].to_string(index=False))
print("\n▼ Categorías con mayor MAE (YOLO falla más):")
print(cat_stats.tail(3)[["categoria", "n", "mae", "accuracy"]].to_string(index=False))

In [ ]:
# ── Sensibilidad al umbral de confianza ──────────────────────────────────────
print("\n── Sensibilidad al umbral de confianza ──")
print(f"{'Umbral':10s} {'Accuracy':>10s} {'MAE':>8s} {'Pred_media':>12s}")
print("-" * 45)

for conf_val, col in [(CONF_MAIN, "pred_low"), (CONF_MEDIUM, "pred_mid"), (CONF_HIGH, "pred_high")]:
    errs = (df[col] - df["ground_truth"]).abs()
    acc  = (df[col] == df["ground_truth"]).mean()
    mae  = errs.mean()
    pred_m = df[col].mean()
    print(f"  {conf_val:.2f}      {acc:10.4f} {mae:8.2f} {pred_m:12.2f}")

# ¿Cuántas imágenes YOLO detecta 0 objetos con conf alto?
n_zeros_low  = (df["pred_low"]  == 0).sum()
n_zeros_mid  = (df["pred_mid"]  == 0).sum()
n_zeros_high = (df["pred_high"] == 0).sum()
print(f"\nImágenes con 0 detecciones:")
print(f"  conf={CONF_MAIN}  → {n_zeros_low}/{len(df)} ({100*n_zeros_low/len(df):.1f}%)")
print(f"  conf={CONF_MEDIUM} → {n_zeros_mid}/{len(df)} ({100*n_zeros_mid/len(df):.1f}%)")
print(f"  conf={CONF_HIGH}  → {n_zeros_high}/{len(df)} ({100*n_zeros_high/len(df):.1f}%)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualizar_prediccion(img_path, categoria, gt, row_data, conf=CONF_MAIN, ax=None):
    """Muestra imagen con bboxes YOLO y título GT vs pred."""
    img = Image.open(img_path).convert("RGB")
    r = predecir_yolo_world(img_path, categoria, conf)

    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(6, 5))

    ax.imshow(img)
    for box in r["boxes_xyxy"]:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=1.5, edgecolor="lime", facecolor="none"
        )
        ax.add_patch(rect)

    color = "green" if r["pred"] == gt else "red"
    ax.set_title(
        f"{categoria}\nGT={gt}  |  YOLO={r['pred']} boxes  (conf≥{conf})",
        fontsize=9, color=color
    )
    ax.axis("off")
    return r

# ── 3 casos exitosos + 3 fallidos (umbral principal) ─────────────────────────
df_sorted = df.sort_values("error_abs")
exitosos  = df_sorted.head(6)
fallidos  = df_sorted.tail(6)

print("── Casos exitosos (menor error absoluto) ──")
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for idx, (ax, (_, row)) in enumerate(zip(axes.flatten(), exitosos.iterrows())):
    path = img_lookup.get(row["imagen"])
    if path:
        visualizar_prediccion(path, row["categoria"], row["ground_truth"], row, ax=ax)
plt.suptitle(f"YOLO-World ({MODEL_TAG}) — Casos con menor error | conf={CONF_MAIN}", fontsize=12)
plt.tight_layout()
plt.savefig("/content/yolo_capture_exitosos.png", dpi=120, bbox_inches="tight")
plt.show()

print("\n── Casos fallidos (mayor error absoluto) ──")
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (_, row) in zip(axes.flatten(), fallidos.iterrows()):
    path = img_lookup.get(row["imagen"])
    if path:
        visualizar_prediccion(path, row["categoria"], row["ground_truth"], row, ax=ax)
plt.suptitle(f"YOLO-World ({MODEL_TAG}) — Casos con mayor error | conf={CONF_MAIN}", fontsize=12)
plt.tight_layout()
plt.savefig("/content/yolo_capture_fallidos.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ── Tabla comparativa YOLO vs VLMs (valores reales de las corridas anteriores) ─
print("\n── Comparativa YOLO-World vs VLMs — CAPTURe real (n=50) ──")

VLM_RESULTS = [
    ("Moondream2 (1.8B)",   0.0000, 47.80,  5114, "n=50*"),
    ("InternVL2-1B",        0.0200, 41.90,  2350, "n=50"),
    ("PaliGemma2-3B",       0.0000, 32.28,  6099, "n=50"),
    ("Qwen2-VL-2B",         0.1000, 23.56,  4785, "n=50"),
]

acc_yolo = (df["pred_low"] == df["ground_truth"]).mean()
mae_yolo = (df["pred_low"] - df["ground_truth"]).abs().mean()

print(f"{'Modelo':22s} {'Accuracy':>10s} {'MAE':>8s} {'VRAM':>8s} {'Nota':>10s}")
print("-" * 65)

yolo_line = f"  {'YOLO-World-M (0-param)':22s} {acc_yolo:10.4f} {mae_yolo:8.2f} {'~0 MB':>8s} {'n=50':>10s}  ← sin GPU requerida"
print(yolo_line)
print()
for nombre, acc, mae, vram, nota in sorted(VLM_RESULTS, key=lambda x: x[2]):
    print(f"  {nombre:22s} {acc:10.4f} {mae:8.2f} {vram:8d} {nota:>10s}")

print("\n*Moondream2 n=50 es el subset comparable; tiene n=824 completo con accuracy=0.04, MAE=50.20")
print(f"\nConf umbral usado para YOLO: {CONF_MAIN}")

In [ ]:
# ── Guardar resultados ──────────────────────────────────────────────────────
acc_main = (df["pred_low"] == df["ground_truth"]).mean()
mae_main = (df["pred_low"] - df["ground_truth"]).abs().mean()
var_main = (df["pred_low"] - df["ground_truth"]).abs().var()

with open(OUTPUT_TXT_PATH, "w", encoding="utf-8") as f:
    f.write(f"BENCHMARK: {BENCHMARK_NAME}\n")
    f.write(f"MODELO: {MODEL_TAG} (YOLO-World open-vocabulary)\n")
    f.write(f"FECHA: {datetime.datetime.now().isoformat()}\n")
    f.write(f"MUESTRA_REAL: {len(df)}/824\n")
    f.write(f"SEMILLA: {RANDOM_SEED}\n")
    f.write(f"CONF_PRINCIPAL: {CONF_MAIN}\n")
    f.write(f"IOU_NMS: {IOU_NMS}\n")
    f.write("-" * 50 + "\n")
    f.write("[REAL]\n")
    f.write(f"  N: {len(df)}\n")
    f.write(f"  ACCURACY_EXACTA: {acc_main:.4f}\n")
    f.write(f"  MAE: {mae_main:.4f}\n")
    f.write(f"  VARIANZA_ERROR: {var_main:.4f}\n")

    # Wilson CI accuracy
    z = 1.96; n = len(df); acc = acc_main
    denom = 1 + z**2/n
    center = (acc + z**2/(2*n)) / denom
    margin_w = (z * np.sqrt(acc*(1-acc)/n + z**2/(4*n**2))) / denom
    f.write(f"  CI_ACCURACY_95_WILSON: [{max(0,center-margin_w):.4f}, {min(1,center+margin_w):.4f}]\n")

    # Bootstrap CI MAE
    errs_v = (df["pred_low"] - df["ground_truth"]).abs().values
    np.random.seed(42)
    bmae = [np.random.choice(errs_v, size=len(errs_v), replace=True).mean() for _ in range(2000)]
    f.write(f"  CI_MAE_95_BOOTSTRAP: [{np.percentile(bmae, 2.5):.2f}, {np.percentile(bmae, 97.5):.2f}]\n")
    f.write("-" * 50 + "\n")

    # Sensibilidad umbral
    f.write("SENSIBILIDAD_UMBRAL\n")
    for conf_val, col in [(CONF_MAIN, "pred_low"), (CONF_MEDIUM, "pred_mid"), (CONF_HIGH, "pred_high")]:
        e = (df[col] - df["ground_truth"]).abs()
        f.write(f"  conf={conf_val}: accuracy={( df[col]==df['ground_truth']).mean():.4f} mae={e.mean():.4f}\n")
    f.write("-" * 50 + "\n")

    # Bucket
    f.write("DESGLOSE_REAL_POR_BUCKET_MAGNITUD\n")
    for bucket in ["8-15", "16-28", "29-67", "68-907"]:
        sub = df[df["bucket"] == bucket]
        if len(sub) == 0:
            continue
        e = (sub["pred_low"] - sub["ground_truth"]).abs()
        f.write(f"  {bucket}: n={len(sub)} accuracy={( sub['pred_low']==sub['ground_truth']).mean():.4f} mae={e.mean():.4f}\n")

print(f"✓ Guardado: {OUTPUT_TXT_PATH}")
print(open(OUTPUT_TXT_PATH).read())

# CSV
csv_cols = ["imagen", "categoria", "ground_truth", "bucket",
            "pred_low", "pred_mid", "pred_high",
            "prediccion", "error_abs", "parse_fail", "timeout", "clase_usada"]
df[csv_cols].to_csv(OUTPUT_CSV_PATH, index=False)
print(f"\n✓ Guardado: {OUTPUT_CSV_PATH}")

In [ ]:
from google.colab import files

files.download(OUTPUT_TXT_PATH)
files.download(OUTPUT_CSV_PATH)
files.download("/content/yolo_capture_exitosos.png")
files.download("/content/yolo_capture_fallidos.png")
print("Archivos descargados.")